# Nigeria Fintech Compliance Quickstart
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comply54/comply54/blob/main/notebooks/01_nigeria_fintech.ipynb)


This notebook demonstrates how to evaluate common Nigerian fintech AI-agent actions with `comply54`.

The `NigeriaFintechCompliance` sector pack combines regulatory and safety controls covering:

- Nigeria Data Protection Act 2023 (NDPA)
- CBN transaction limits and controls
- CBN BVN Framework and NIBSS rules
- MLPPA 2022 / NFIU AML guidelines
- OWASP Agentic AI safety controls

We'll evaluate several scenarios to see how comply54 allows, escalates, or denies actions before they are executed.

In [ ]:
%pip install comply54

In [ ]:
from comply54.sectors import NigeriaFintechCompliance

compliance = NigeriaFintechCompliance()

print(compliance.name)
print("Jurisdictions:", compliance.jurisdictions)
print("Regulations:")
for regulation in compliance.regulations:
    print(f"  - {regulation}")

In [ ]:
def show_result(result):
    print(f"Overall decision: {result.overall.upper()}")
    print(f"Blocked: {result.blocked}")
    print(f"Audit ID: {result.audit_id}")
    print(f"Evaluated at: {result.evaluated_at}")

    notable_decisions = [
        decision for decision in result.decisions
        if decision.action != "allow"
    ]

    if notable_decisions:
        print("\nPolicy decisions:")
        for decision in notable_decisions:
            print(f"\n  Pack: {decision.pack}")
            print(f"  Regulation: {decision.regulation}")
            print(f"  Decision: {decision.action.upper()}")

            for message in decision.messages:
                print(f"  Message: {message}")

            if decision.citations:
                print("  Citations:")
                for citation in decision.citations:
                    print(
                        f"    - {citation.document}, "
                        f"{citation.section} ({citation.authority}, {citation.year})"
                    )
    else:
        print("\nNo non-allow policy decisions were triggered.")

## Scenario 1 — Small screened transfer

A fully verified customer makes a small domestic transfer after sanctions screening.

The transaction is permitted to proceed, while comply54 records an `audit` decision for compliance traceability.

In [ ]:
small_transfer = compliance.check(
    action="transfer_funds",
    params={
        "amount": 5_000,
        "currency": "NGN",
    },
    context={
        "kyc_tier": 3,
        "sanctions_screened": True,
        "aml_check_performed": True,
        "kyc_verified": True,
    },
)

show_result(small_transfer)

## Scenario 2 — Transfer above the CBN NIP limit

The customer attempts a ₦15,000,000 transfer.

This exceeds the ₦10,000,000 NIP transaction cap enforced by the CBN policy pack, so the action should be denied.

In [ ]:
large_transfer = compliance.check(
    action="transfer_funds",
    params={
        "amount": 15_000_000,
        "currency": "NGN",
    },
    context={
        "kyc_tier": 3,
        "sanctions_screened": True,
    },
)

show_result(large_transfer)

## Scenario 3 — High-value transaction requiring escalation

A screened customer attempts a ₦6,000,000 transfer.

The transaction is below the ₦10,000,000 NIP cap, but its value can trigger additional compliance review. This demonstrates an `escalate` decision rather than an outright denial.

In [ ]:
high_value_transfer = compliance.check(
    action="transfer_funds",
    params={
        "amount": 6_000_000,
        "currency": "NGN",
    },
    context={
        "kyc_tier": 3,
        "sanctions_screened": True,
    },
)

show_result(high_value_transfer)

## Scenario 4 — Transfer without sanctions screening

A transfer is attempted without completing the required sanctions screening.

Even though the amount is below the NIP transaction cap, the missing compliance control causes the action to be blocked.

In [ ]:
unscreened_transfer = compliance.check(
    action="transfer_funds",
    params={
        "amount": 3_000_000,
        "currency": "NGN",
    },
    context={
        "sanctions_screened": False,
        "aml_check_performed": False,
    },
)

show_result(unscreened_transfer)

## What happens next — LangGraph integration

Direct compliance checks are useful for evaluating individual actions, but in an AI-agent workflow you typically want compliance enforcement to happen automatically before a tool executes.

comply54 provides `Comply54Guard`, which can be inserted into a LangGraph workflow between the agent and its tools:

```python
from comply54.sectors import NigeriaFintechCompliance
from comply54.langchain import Comply54Guard, comply54_route
from langgraph.graph import END, StateGraph
from langgraph.prebuilt import ToolNode

compliance = NigeriaFintechCompliance()

guard = Comply54Guard(
    compliance=compliance,
    context={"kyc_tier": 3, "sanctions_screened": True},
)

graph = StateGraph(AgentState)

graph.add_node("agent", call_model)
graph.add_node("comply54_guard", guard)
graph.add_node("tools", ToolNode(tools))

graph.add_conditional_edges(
    "comply54_guard",
    comply54_route,
    {"tools": "tools", "agent": "agent"},
)
```

The guard evaluates proposed tool calls before they reach the tool node. Actions that violate the configured compliance policies can therefore be stopped or routed for further handling before execution.

> **Note:** The core examples in this notebook require only `comply54`. Running the LangGraph integration itself requires the optional LangGraph dependencies (`comply54[langgraph]`).